In [1]:
import pandas as pd
import numpy
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('/content/job_dataset_20_roles_20000.csv')
pd.set_option('display.max_columns', None)

clean = df.drop(columns=['Job ID', 'Job Description','Job Title'])
#clean.head()
df.isnull().sum()

,0
Job ID,0
Job Title,0
Job Role,0
Job Level,0
Domain,0
Tech Stack,0
Skills,0
Required Skills,0
Experience (Years),0
Education,0


In [2]:
a = clean.isnull().sum()
clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 19 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Job Role              20000 non-null  object
 1   Job Level             20000 non-null  object
 2   Domain                20000 non-null  object
 3   Tech Stack            20000 non-null  object
 4   Skills                20000 non-null  object
 5   Required Skills       20000 non-null  object
 6   Experience (Years)    20000 non-null  int64 
 7   Education             20000 non-null  object
 8   Certifications        14998 non-null  object
 9   Certifications Count  20000 non-null  int64 
 10  Projects Count        20000 non-null  int64 
 11  Salary (USD/Year)     20000 non-null  int64 
 12  Employment Type       20000 non-null  object
 13  Work Mode             20000 non-null  object
 14  Company Name          20000 non-null  object
 15  Company Size          20000 non-null

In [3]:
#Search for most Null data (Certifications)
NoCer = clean[clean['Certifications'].isna()]

#NoCer.head()

# Search for job without Experience
NoEx = clean[clean['Experience (Years)'] == 0]
# NoEx.head()

# Self taught Education
Selftaught = clean[clean['Education'] == "Bootcamp + Self-Taught"]

In [4]:
import pandas as pd
import plotly.graph_objects as go
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

SAL = 'Salary (USD/Year)'

# 1. Aggregate: country level + city level
avg_salary_country = (
    clean.groupby('Country', as_index=False)[SAL].mean().round(2)
    .rename(columns={SAL: 'Country_Avg_Salary'})
)

avg_salary_city = (
    clean.groupby(['Country', 'City'], as_index=False)
    .agg(City_Avg_Salary=(SAL, 'mean'), Postings_Count=(SAL, 'count'))
    .round(2)
)

# 2. Geocode cities (cached, rate-limited)
geolocator = Nominatim(user_agent="salary_map_visualizer")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1, max_retries=2, error_wait_seconds=3)

cache = {}
def get_coords(city, country):
    key = (city, country)
    if key not in cache:
        try:
            loc = geocode(f"{city}, {country}", timeout=10)
            cache[key] = (loc.latitude, loc.longitude) if loc else (None, None)
        except Exception:
            cache[key] = (None, None)
    return cache[key]

coords = [get_coords(c, k) for c, k in zip(avg_salary_city['City'], avg_salary_city['Country'])]
avg_salary_city['Latitude'] = [c[0] for c in coords]
avg_salary_city['Longitude'] = [c[1] for c in coords]

valid_cities = avg_salary_city.dropna(subset=['Latitude', 'Longitude']).copy()
print(f"Geocoded {len(valid_cities)} / {len(avg_salary_city)} cities.")

# 3. Country hover text now lists its cities (ranked by avg salary)
def city_breakdown(g, n=6):
    g = g.sort_values('City_Avg_Salary', ascending=False)
    lines = [f"{i+1}. {r.City}: ${r.City_Avg_Salary:,.0f} ({r.Postings_Count} jobs)"
             for i, r in enumerate(g.head(n).itertuples())]
    if len(g) > n:
        lines.append(f"... +{len(g) - n} more cities")
    return "<br>".join(lines)

breakdown = (avg_salary_city.groupby('Country')
             .apply(city_breakdown, include_groups=False)
             .rename('City_List').reset_index())
avg_salary_country = avg_salary_country.merge(breakdown, on='Country', how='left')

# 4. Marker sizing (global, so sizes stay consistent when filtering)
min_sal, max_sal = valid_cities['City_Avg_Salary'].min(), valid_cities['City_Avg_Salary'].max()
def size_of(s):
    return 8 + (s - min_sal) / (max_sal - min_sal) * 16 if max_sal > min_sal else 12

valid_cities['Size'] = valid_cities['City_Avg_Salary'].apply(size_of)
valid_cities['Label'] = valid_cities['City'] + " (" + valid_cities['Country'] + ")"

def city_payload(df_):
    return {
        'lat': [df_['Latitude'].tolist()],
        'lon': [df_['Longitude'].tolist()],
        'text': [df_['Label'].tolist()],
        'customdata': [df_[['City_Avg_Salary', 'Postings_Count']].values.tolist()],
        'marker.size': [df_['Size'].tolist()],
        'marker.color': [df_['City_Avg_Salary'].tolist()],
    }

# 5. Build figure
fig = go.Figure()

fig.add_trace(go.Choropleth(
    locations=avg_salary_country['Country'],
    locationmode='country names',
    z=avg_salary_country['Country_Avg_Salary'],
    customdata=avg_salary_country[['City_List']],
    text=avg_salary_country['Country'],
    colorscale='Blues',
    colorbar=dict(title="Country Avg ($)", x=1.02),
    hovertemplate="<b>%{text}</b><br>Country Avg Salary: $%{z:,.0f}"
                  "<br><br><b>Cities (ranked by avg salary):</b><br>%{customdata[0]}<extra></extra>"
))

fig.add_trace(go.Scattergeo(
    lon=valid_cities['Longitude'],
    lat=valid_cities['Latitude'],
    text=valid_cities['Label'],
    customdata=valid_cities[['City_Avg_Salary', 'Postings_Count']].values,
    mode='markers',
    marker=dict(
        size=valid_cities['Size'],
        color=valid_cities['City_Avg_Salary'],
        cmin=min_sal, cmax=max_sal,
        colorscale='Reds',
        colorbar=dict(title="City Avg ($)", x=1.14),
        showscale=True,
        line=dict(width=1.5, color='black'),
        opacity=0.9
    ),
    hovertemplate="<b>%{text}</b><br>City Avg Salary: $%{customdata[0]:,.0f}"
                  "<br>Open Vacancies: %{customdata[1]}<extra></extra>"
))

# 6. Dropdown: All countries, or zoom into one country and show only its cities
def zoom_range(df_, pad=4):
    lat_min, lat_max = df_['Latitude'].min(), df_['Latitude'].max()
    lon_min, lon_max = df_['Longitude'].min(), df_['Longitude'].max()
    p_lat = max(pad, (lat_max - lat_min) * 0.3)
    p_lon = max(pad, (lon_max - lon_min) * 0.3)
    return ([max(lon_min - p_lon, -180), min(lon_max + p_lon, 180)],
            [max(lat_min - p_lat, -90), min(lat_max + p_lat, 90)])

buttons = [dict(
    label='All countries',
    method='update',
    args=[city_payload(valid_cities) | {},  # data update applies to trace 1 only
          {'geo.lonaxis.range': [-180, 180], 'geo.lataxis.range': [-90, 90],
           'title': 'Interactive Salary Map: Country Average + City Detail (USD/Year)'}],
)]
buttons[0]['args'][0] = city_payload(valid_cities)

for country in sorted(valid_cities['Country'].unique()):
    sub = valid_cities[valid_cities['Country'] == country]
    lon_r, lat_r = zoom_range(sub)
    buttons.append(dict(
        label=country,
        method='update',
        args=[city_payload(sub),
              {'geo.lonaxis.range': lon_r, 'geo.lataxis.range': lat_r,
               'title': f'Salary by City - {country} (USD/Year)'}],
    ))

# Apply the data update only to the city trace (index 1)
for b in buttons:
    b['args'] = [b['args'][0], b['args'][1], [1]]

fig.update_layout(
    title='Interactive Salary Map: Country Average + City Detail (USD/Year)',
    updatemenus=[dict(
        buttons=buttons, direction='down', showactive=True,
        x=0.0, xanchor='left', y=1.12, yanchor='top'
    )],
    geo=dict(
        showframe=False,
        showcoastlines=True,
        showcountries=True,
        countrycolor="rgb(180, 180, 180)",
        projection_type='equirectangular',
        lonaxis=dict(range=[-180, 180]),
        lataxis=dict(range=[-90, 90]),
    ),
    margin=dict(l=0, r=0, t=90, b=0),
    height=750
)

fig.show()

Geocoded 28 / 28 cities.


In [5]:
# ==============================================================================
# AVERAGE SALARY BY DOMAIN: HORIZONTAL BAR WITH YEAR FILTER & AUTO-SORT
# ==============================================================================

import pandas as pd
import plotly.graph_objects as go

# 1. Clean and prepare data
df_domain = clean.dropna(subset=['Domain', 'Salary (USD/Year)', 'Date Posted']).copy()

# Ensure numeric salary
if df_domain['Salary (USD/Year)'].dtype == 'object':
    df_domain['Salary (USD/Year)'] = (
        df_domain['Salary (USD/Year)']
        .astype(str)
        .str.replace(r'[$,]', '', regex=True)
    )
df_domain['Salary (USD/Year)'] = pd.to_numeric(df_domain['Salary (USD/Year)'], errors='coerce')
df_domain = df_domain.dropna(subset=['Salary (USD/Year)'])

# Extract and filter years (2024 & 2025)
df_domain['Date Posted'] = pd.to_datetime(df_domain['Date Posted'], errors='coerce')
df_domain['Posting Year'] = df_domain['Date Posted'].dt.year
df_domain = df_domain[df_domain['Posting Year'].isin([2024, 2025])]
df_domain['Posting Year'] = df_domain['Posting Year'].astype(int).astype(str)

# 2. Pivot table: Index = Domain, Columns = 2024, 2025
domain_pivot = (
    df_domain.groupby(['Domain', 'Posting Year'])['Salary (USD/Year)']
    .mean()
    .round(2)
    .unstack(fill_value=0)
    .reset_index()
)

for yr in ['2024', '2025']:
    if yr not in domain_pivot.columns:
        domain_pivot[yr] = 0

# Overall average across both years for combined sorting
domain_pivot['Overall'] = domain_pivot[['2024', '2025']].replace(0, pd.NA).mean(axis=1).fillna(0).round(2)

# Helper function: Sort ascending for Plotly horizontal bars (highest appears on top)
def get_sorted_data(sort_col):
    sorted_df = domain_pivot.sort_values(by=sort_col, ascending=True)
    domains = sorted_df['Domain'].tolist()
    vals_24 = sorted_df['2024'].tolist()
    vals_25 = sorted_df['2025'].tolist()
    return domains, vals_24, vals_25

d_all, v24_all, v25_all = get_sorted_data('Overall')
d_24, v24_only, _ = get_sorted_data('2024')
d_25, _, v25_only = get_sorted_data('2025')

# 3. Build Figure with Horizontal Traces
fig = go.Figure()

# Trace 0: 2024
fig.add_trace(go.Bar(
    y=d_all,
    x=v24_all,
    name='2024 Avg Salary',
    orientation='h',
    marker_color='#4C78A8',
    text=[f"${v:,.0f}" if v > 0 else "N/A" for v in v24_all],
    textposition='outside',
    hovertemplate="<b>Domain:</b> %{y}<br>2024 Avg Salary: $%{x:,.2f}<extra></extra>"
))

# Trace 1: 2025
fig.add_trace(go.Bar(
    y=d_all,
    x=v25_all,
    name='2025 Avg Salary',
    orientation='h',
    marker_color='#F58518',
    text=[f"${v:,.0f}" if v > 0 else "N/A" for v in v25_all],
    textposition='outside',
    hovertemplate="<b>Domain:</b> %{y}<br>2025 Avg Salary: $%{x:,.2f}<extra></extra>"
))

# 4. Interactive Dropdown Menu (Filter + Auto Re-sort High-to-Low)
buttons = [
    dict(
        label='Both Years (Compare 2024 vs 2025)',
        method='update',
        args=[
            {
                'y': [d_all, d_all],
                'x': [v24_all, v25_all],
                'text': [[f"${v:,.0f}" if v > 0 else "N/A" for v in v24_all],
                         [f"${v:,.0f}" if v > 0 else "N/A" for v in v25_all]],
                'visible': [True, True]
            },
            {
                'title': 'Average Tech Salary by Domain: 2024 vs 2025 (Sorted by Overall Avg)',
                'barmode': 'group'
            }
        ]
    ),
    dict(
        label='Only 2024 (Ranked High to Low)',
        method='update',
        args=[
            {
                'y': [d_24, []],
                'x': [v24_only, []],
                'text': [[f"${v:,.0f}" if v > 0 else "N/A" for v in v24_only], []],
                'visible': [True, False]
            },
            {
                'title': 'Average Tech Salary by Domain in 2024 (Highest to Lowest)',
                'barmode': 'group'
            }
        ]
    ),
    dict(
        label='Only 2025 (Ranked High to Low)',
        method='update',
        args=[
            {
                'y': [[], d_25],
                'x': [[], v25_only],
                'text': [[], [f"${v:,.0f}" if v > 0 else "N/A" for v in v25_only]],
                'visible': [False, True]
            },
            {
                'title': 'Average Tech Salary by Domain in 2025 (Highest to Lowest)',
                'barmode': 'group'
            }
        ]
    )
]

# 5. Layout Setup
fig.update_layout(
    updatemenus=[dict(
        active=0,
        buttons=buttons,
        direction="down",
        pad={"r": 10, "t": 10},
        showactive=True,
        x=0.0,
        xanchor="left",
        y=1.18,
        yanchor="top"
    )],
    title='Average Tech Salary by Domain: 2024 vs 2025 (Sorted by Overall Avg)',
    xaxis_title='Average Salary (USD/Year)',
    yaxis_title='Domain',
    barmode='group',
    template='plotly_white',
    height=max(550, len(domain_pivot) * 45),
    margin=dict(t=100, b=60, l=150, r=60),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

fig.show()

In [6]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "colab"

TOP = 20  # change this number to show more/fewer skills

# 1. Unpack skills + year
t = df[['Required Skills', 'Date Posted']].dropna().copy()
t['Date Posted'] = pd.to_datetime(t['Date Posted'], errors='coerce')
t = t.dropna(subset=['Date Posted'])
t['Year'] = t['Date Posted'].dt.year
t = t[t['Year'].isin([2024, 2025])]
t['Year'] = t['Year'].astype(str)
t['Skill'] = t['Required Skills'].astype(str).apply(
    lambda x: [i.strip() for i in x.split('|') if i.strip()])
t = t.explode('Skill')

sy = t.groupby(['Skill', 'Year']).size().unstack(fill_value=0)
for yr in ['2024', '2025']:
    if yr not in sy.columns:
        sy[yr] = 0
sy = sy[['2024', '2025']]
sy['Total'] = sy['2024'] + sy['2025']
sy['Change'] = sy['2025'] - sy['2024']
print("Postings by year:", t.groupby('Year').size().to_dict())

top = sy.sort_values('Total', ascending=False).head(TOP)
order_both = top.sort_values('Total').index.tolist()      # bottom -> top
order_24 = top.sort_values('2024').index.tolist()
order_25 = top.sort_values('2025').index.tolist()

COLORS = {'2024': '#1f77b4', '2025': '#ff7f0e'}
fig = go.Figure()
for yr in ['2024', '2025']:
    fig.add_trace(go.Bar(
        y=top.index, x=top[yr], orientation='h', name=yr, marker_color=COLORS[yr],
        text=top[yr], textposition='outside', customdata=top['Change'],
        hovertemplate='%{y}<br>' + yr + ': %{x} postings<br>Change 2024→2025: %{customdata:+d}<extra></extra>'))

def btn(label, vis, order, title):
    return dict(label=label, method='update',
                args=[{'visible': vis},
                      {'yaxis.categoryarray': order, 'yaxis.categoryorder': 'array', 'title.text': title}])

fig.update_layout(
    title=f'Top {TOP} Most Required Skills - 2024 vs 2025',
    barmode='group', template='plotly_white',
    xaxis_title='Number of Job Postings Requiring the Skill', yaxis_title='Skill',
    yaxis=dict(categoryorder='array', categoryarray=order_both),
    height=max(500, 150 + 32 * TOP * 2), margin=dict(t=110, b=50, l=40, r=60),
    updatemenus=[dict(
        buttons=[btn('Both years', [True, True], order_both, f'Top {TOP} Most Required Skills - 2024 vs 2025'),
                 btn('2024 only', [True, False], order_24, f'Top {TOP} Most Required Skills - 2024'),
                 btn('2025 only', [False, True], order_25, f'Top {TOP} Most Required Skills - 2025')],
        direction='down', x=0, xanchor='left', y=1.15, yanchor='top', showactive=True)])
fig.show()

Postings by year: {'2024': 49615, '2025': 50385}


In [7]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "colab"

TOP = 10       # skills shown per role
TOP_ALL = 15   # skills shown for "All jobs"
ALL_JOBS = 'All jobs'

# 1. Unpack skills + year
t = df[['Job Role', 'Required Skills', 'Date Posted']].dropna().copy()
t['Date Posted'] = pd.to_datetime(t['Date Posted'], errors='coerce')
t = t.dropna(subset=['Date Posted'])
t['Year'] = t['Date Posted'].dt.year
t = t[t['Year'].isin([2024, 2025])]
t['Year'] = t['Year'].astype(str)
t['Skill'] = t['Required Skills'].astype(str).apply(
    lambda x: [i.strip() for i in x.split('|') if i.strip()])
t = t.explode('Skill')

# 2. Counts helper -> columns 2024, 2025, Total, Change
def make_counts(g):
    c = g.unstack(fill_value=0)
    for yr in ['2024', '2025']:
        if yr not in c.columns:
            c[yr] = 0
    c = c[['2024', '2025']]
    c['Total'] = c['2024'] + c['2025']
    c['Change'] = c['2025'] - c['2024']
    return c

counts_role = make_counts(t.groupby(['Job Role', 'Skill', 'Year']).size())   # Job Role x Skill
counts_all = make_counts(t.groupby(['Skill', 'Year']).size())                # All jobs: Skill only

# postings per role/year (+ overall)
d = df.assign(_d=pd.to_datetime(df['Date Posted'], errors='coerce')).dropna(subset=['_d'])
d['Year'] = d['_d'].dt.year.astype(str)
postings = d.groupby(['Job Role', 'Year']).size()
postings_all = d.groupby('Year').size()

roles = sorted(counts_role.index.get_level_values('Job Role').unique())
options = [ALL_JOBS] + roles
COLORS = {'2024': '#1f77b4', '2025': '#ff7f0e'}

# 3. Metadata + traces (2 traces per option)
fig = go.Figure()
meta = {}
for i, opt in enumerate(options):
    if opt == ALL_JOBS:
        data, n = counts_all.sort_values('Total', ascending=False).head(TOP_ALL), TOP_ALL
        n24, n25 = int(postings_all.get('2024', 0)), int(postings_all.get('2025', 0))
        label = 'All Jobs'
    else:
        data, n = counts_role.loc[opt].sort_values('Total', ascending=False).head(TOP), TOP
        n24, n25 = int(postings.get((opt, '2024'), 0)), int(postings.get((opt, '2025'), 0))
        label = f"'{opt}'"

    order = data.sort_values('Total').index.tolist()                   # bottom -> top
    xmax = max(data[['2024', '2025']].values.max() * 1.15, 1)
    ch = data.sort_values('Change', ascending=False)
    sub = f"Postings: 2024 = {n24}, 2025 = {n25}"
    if len(ch) > 1:
        sub += (f" | Biggest rise: {ch.index[0]} ({ch['Change'].iloc[0]:+d})"
                f" | Biggest drop: {ch.index[-1]} ({ch['Change'].iloc[-1]:+d})")
    meta[opt] = dict(order=order, xmax=xmax,
                     title=f"Top {n} Skills for {label} - 2024 vs 2025<br><sup>{sub}</sup>")

    for yr in ['2024', '2025']:
        fig.add_trace(go.Bar(
            y=data.index, x=data[yr], orientation='h', name=yr, marker_color=COLORS[yr],
            text=data[yr], textposition='outside', customdata=data['Change'],
            visible=(i == 0), legendgroup=yr, showlegend=(i == 0),
            hovertemplate='%{y}<br>' + yr + ': %{x} postings'
                          '<br>Change 2024→2025: %{customdata:+d}<extra></extra>'))

# 4. Dropdown buttons
buttons = []
for i, opt in enumerate(options):
    vis = [False] * (2 * len(options))
    vis[2 * i] = vis[2 * i + 1] = True
    buttons.append(dict(
        label=opt, method='update',
        args=[{'visible': vis, 'showlegend': vis},
              {'title.text': meta[opt]['title'],
               'yaxis.categoryarray': meta[opt]['order'],
               'xaxis.range': [0, meta[opt]['xmax']]}]))

first = meta[ALL_JOBS]
fig.update_layout(
    title=first['title'], barmode='group', template='plotly_white',
    xaxis=dict(title='Number of Job Postings Requiring the Skill', range=[0, first['xmax']]),
    yaxis=dict(title='Skill', categoryorder='array', categoryarray=first['order']),
    height=max(500, 160 + 32 * max(TOP, TOP_ALL) * 2), margin=dict(t=130, b=50, l=40, r=60),
    legend=dict(orientation='h', y=1.02, x=1, xanchor='right'),
    updatemenus=[dict(buttons=buttons, direction='down', x=0, xanchor='left',
                      y=1.18, yanchor='top', showactive=True)])
fig.show()

In [13]:
# JOB VACANCIES - INTERACTIVE (Year buttons + Job Role dropdown)

import traceback
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import ipywidgets as widgets
from IPython.display import display

pio.renderers.default = "colab"

# 1. Clean & prepare data (uses your `clean` DataFrame)
df_trend = clean.dropna(subset=['Date Posted', 'Job Role']).copy()
df_trend['Job Role'] = df_trend['Job Role'].astype(str).str.strip()   # SORT FIX: force text
df_trend = df_trend[df_trend['Job Role'] != '']
df_trend['Date Posted'] = pd.to_datetime(df_trend['Date Posted'], errors='coerce')
df_trend = df_trend.dropna(subset=['Date Posted'])
df_trend['Posting Year'] = df_trend['Date Posted'].dt.year
df_trend = df_trend[df_trend['Posting Year'].isin([2024, 2025])]
df_trend['Posting Year'] = df_trend['Posting Year'].astype(int).astype(str)

# 2. Pivot: index = Job Role, columns = 2024 / 2025
pivot = (df_trend.groupby(['Job Role', 'Posting Year']).size()
         .unstack(fill_value=0))
for yr in ['2024', '2025']:
    if yr not in pivot.columns:
        pivot[yr] = 0
pivot = pivot[['2024', '2025']]
pivot['Total'] = pivot['2024'] + pivot['2025']
print("Roles:", len(pivot), "| Postings:", int(pivot['Total'].sum()))

COLORS = {'2024': '#1f77b4', '2025': '#ff7f0e'}
ALL_YEARS = 'Both years (2024 vs 2025)'
ALL_JOBS = 'All jobs'

# 3. Draw function
def draw(year, role):
    try:
        data = pivot if role == ALL_JOBS else pivot.loc[[role]]
        years = ['2024', '2025'] if year == ALL_YEARS else [year]

        # SORT FIX: sort by count, ties broken by role name (stable, never errors)
        sort_col = 'Total' if len(years) == 2 else years[0]
        data = (data.assign(_name=data.index.astype(str).str.lower())
                    .sort_values([sort_col, '_name'], ascending=[True, False])
                    .drop(columns='_name'))

        fig = go.Figure()
        for yr in years:
            fig.add_trace(go.Bar(
                y=data.index, x=data[yr], orientation='h', name=yr,
                marker_color=COLORS[yr],
                text=data[yr], textposition='outside',
                hovertemplate='%{y}<br>' + yr + ': %{x} vacancies<extra></extra>'
            ))

        yr_label = '2024 vs 2025' if len(years) == 2 else years[0]
        job_label = 'All Jobs' if role == ALL_JOBS else role
        title = f"Job Vacancies - {job_label} ({yr_label})"
        if role != ALL_JOBS and len(years) == 2:
            v24, v25 = int(data['2024'].iloc[0]), int(data['2025'].iloc[0])
            if v24 == v25:
                title += f"<br><sup>Same demand in both years ({v24})</sup>"
            else:
                hi, diff = ('2025', v25 - v24) if v25 > v24 else ('2024', v24 - v25)
                title += f"<br><sup>{hi} has {diff} more vacancies</sup>"

        max_val = data[years].values.max()
        fig.update_layout(
            title=title,
            barmode='group',
            template='plotly_white',
            xaxis_title='Number of Vacancies',
            yaxis_title='Job Role',
            height=max(350, 90 + 28 * len(data) * len(years)),
            margin=dict(t=90, b=50, l=40, r=40),
            legend=dict(orientation='h', y=1.08, x=1, xanchor='right'),
        )
        fig.update_xaxes(range=[0, max(max_val * 1.15, 1)])
        fig.show()
    except Exception:
        print(traceback.format_exc())   # show errors instead of silently showing nothing

# 4. Widgets (Year = buttons, Job Role = dropdown)
w_year = widgets.ToggleButtons(options=[ALL_YEARS, '2024', '2025'], value=ALL_YEARS,
                               description='Year:', button_style='info')
w_role = widgets.Dropdown(options=[ALL_JOBS] + sorted(pivot.index.tolist(), key=lambda s: s.lower()),
                          value=ALL_JOBS, description='Job Role:',
                          layout=widgets.Layout(width='350px'))

out = widgets.interactive_output(draw, {'year': w_year, 'role': w_role})
display(widgets.VBox([w_year, w_role]), out)

Roles: 20 | Postings: 20000


Output()

In [9]:
# ==============================================================================
# AVG SALARY BY JOB ROLE x JOB LEVEL - Year buttons + Role dropdown
# ==============================================================================
from google.colab import output
output.enable_custom_widget_manager()

import traceback
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display

SAL = 'Salary (USD/Year)'
d = df[['Job Role', 'Job Level', SAL, 'Date Posted']].copy()
d[SAL] = pd.to_numeric(d[SAL], errors='coerce')
d['Date Posted'] = pd.to_datetime(d['Date Posted'], errors='coerce')
d = d.dropna(subset=['Job Role', 'Job Level', SAL])

# FIX: force text so sorting never mixes str / float / int
d['Job Role'] = d['Job Role'].astype(str).str.strip()
d['Job Level'] = d['Job Level'].astype(str).str.strip()
d = d[(d['Job Role'] != '') & (d['Job Level'] != '')]
d['Year'] = d['Date Posted'].dt.year.fillna(0).astype(int)   # 0 = unknown date

years = sorted(int(y) for y in d['Year'].unique() if y > 0)
roles = sorted(d['Job Role'].unique(), key=lambda s: s.lower())

# Level order junior -> senior (edit KNOWN if your level names differ)
KNOWN = ['intern', 'entry', 'entry-level', 'junior', 'mid', 'mid-level', 'intermediate',
         'senior', 'lead', 'staff', 'principal', 'manager', 'director', 'vp', 'executive']
def lvl_key(x):
    k = str(x).strip().lower()
    return (KNOWN.index(k) if k in KNOWN else len(KNOWN), k)   # always (int, str)
LEVELS = sorted(d['Job Level'].unique(), key=lvl_key)
print("Level order used:", LEVELS)
print("Years found:", years, "| Roles:", len(roles))

ALL_ROLES, ALL_YEARS, COMPARE = 'All roles', 'All years', 'Compare years'
PALETTE = px.colors.qualitative.Plotly

def pivot(data, how):
    return (data.pivot_table(index='Job Role', columns='Job Level', values=SAL, aggfunc=how)
                .reindex(index=roles, columns=LEVELS))

def fmt_k(v, sign=False):
    if pd.isna(v): return ''
    return f"{v/1000:+,.0f}k" if sign else f"${v/1000:,.0f}k"

def build_fig(role, year):
    # All roles -> heatmap Role x Level
    if role == ALL_ROLES:
        if year == COMPARE and len(years) >= 2:
            y0, y1 = years[0], years[-1]
            z = pivot(d[d.Year == y1], 'mean') - pivot(d[d.Year == y0], 'mean')
            n = pivot(d[d.Year.isin([y0, y1])], 'count').fillna(0)
            title = f"Salary change by Role x Level: {y0} → {y1} (USD/Year)"
            text = [[fmt_k(v, True) for v in r] for r in z.values]
            scale, zmid = 'RdBu', 0
        else:
            data = d if year in (ALL_YEARS, COMPARE) else d[d.Year == int(year)]
            z, n = pivot(data, 'mean'), pivot(data, 'count').fillna(0)
            label = 'All years' if year in (ALL_YEARS, COMPARE) else year
            title = f"Average Salary by Role x Level - {label} (USD/Year)"
            text = [[fmt_k(v) for v in r] for r in z.values]
            scale, zmid = 'YlGnBu', None

        fig = go.Figure(go.Heatmap(
            z=z.values, x=LEVELS, y=roles, customdata=n.values,
            text=text, texttemplate="%{text}", colorscale=scale, zmid=zmid,
            colorbar=dict(title="USD"),
            hovertemplate="%{y} | %{x}<br>Value: %{z:,.0f}<br>Postings: %{customdata:.0f}<extra></extra>"))
        fig.update_layout(title=title, template='plotly_white',
                          height=max(450, 120 + 35 * len(roles)),
                          xaxis_title='Job Level', yaxis_title='Job Role',
                          yaxis=dict(autorange='reversed'),
                          margin=dict(t=80, b=50, l=40, r=40))
        return fig

    # One role -> horizontal bars per level
    sub = d[d['Job Role'] == role]
    if year == COMPARE:
        series, label = {str(y): sub[sub.Year == y] for y in years}, 'by year'
    elif year == ALL_YEARS:
        series, label = {'All years': sub}, 'All years'
    else:
        series, label = {year: sub[sub.Year == int(year)]}, year

    fig = go.Figure()
    max_v = 1
    for i, (name, s) in enumerate(series.items()):
        g = s.groupby('Job Level')[SAL].agg(['mean', 'count']).reindex(LEVELS)
        if g['mean'].notna().any():
            max_v = max(max_v, g['mean'].max())
        fig.add_trace(go.Bar(
            y=LEVELS, x=g['mean'], orientation='h', name=name,
            marker_color=PALETTE[i % len(PALETTE)],
            text=[fmt_k(v) for v in g['mean']], textposition='outside',
            customdata=g['count'].fillna(0),
            hovertemplate="%{y}<br>Avg: $%{x:,.0f}<br>Postings: %{customdata:.0f}<extra></extra>"))
    fig.update_layout(
        title=f"Average Salary by Job Level - {role} ({label})",
        barmode='group', template='plotly_white',
        xaxis_title='Average Salary (USD/Year)', yaxis_title='Job Level',
        yaxis=dict(autorange='reversed'),
        height=max(400, 120 + 40 * len(LEVELS) * len(series)),
        margin=dict(t=80, b=50, l=40, r=60),
        legend=dict(orientation='h', y=1.08, x=1, xanchor='right'))
    fig.update_xaxes(range=[0, max_v * 1.2])
    return fig

# ---------- Controls ----------
w_year = widgets.ToggleButtons(options=[ALL_YEARS] + [str(y) for y in years] + [COMPARE],
                               value=ALL_YEARS, description='Year:', button_style='info')
w_role = widgets.Dropdown(options=[ALL_ROLES] + roles, value=ALL_ROLES, description='Job Role:',
                          layout=widgets.Layout(width='400px'))
chart_box = widgets.VBox()
err_box = widgets.Output()   # callback errors are printed here instead of being hidden

def update(_=None):
    err_box.clear_output()
    try:
        chart_box.children = [go.FigureWidget(build_fig(w_role.value, w_year.value))]
    except Exception:
        with err_box:
            print(traceback.format_exc())

w_year.observe(update, names='value')
w_role.observe(update, names='value')

update()
display(widgets.VBox([w_year, w_role, chart_box, err_box]))

Level order used: ['Junior', 'Mid-Level', 'Senior', 'Lead', 'Principal / Staff']
Years found: [2024, 2025] | Roles: 20


In [10]:

#  DO EXPERIENCE / JOB LEVEL / JOB ROLE AFFECT SALARY?

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from scipy import stats

pio.renderers.default = "colab"

SAL, EXP = 'Salary (USD/Year)', 'Experience (Years)'
c = df[['Job Role', 'Job Level', EXP, SAL]].copy()
c[SAL] = pd.to_numeric(c[SAL], errors='coerce')
c[EXP] = pd.to_numeric(c[EXP], errors='coerce')
c = c.dropna()

# Ordinal-encode Job Level (junior -> senior). Edit KNOWN if your level names differ.
KNOWN = ['intern', 'entry', 'entry-level', 'junior', 'mid', 'mid-level', 'intermediate',
         'senior', 'lead', 'staff', 'principal', 'manager', 'director', 'vp', 'executive']
def lvl_key(x):
    k = x.strip().lower()
    return (KNOWN.index(k), x) if k in KNOWN else (len(KNOWN), x)
LEVELS = sorted(c['Job Level'].unique(), key=lvl_key)
c['Level_Code'] = c['Job Level'].map({l: i for i, l in enumerate(LEVELS)})
print("Level order (0 = lowest):", LEVELS)

# Correlation ratio (eta): strength of a categorical variable vs a numeric one
def eta(cat, y):
    grand = y.mean()
    ss_between = sum(len(g) * (g.mean() - grand) ** 2 for _, g in y.groupby(cat))
    ss_total = ((y - grand) ** 2).sum()
    return float(np.sqrt(ss_between / ss_total)) if ss_total > 0 else np.nan

def strength(v):
    v = abs(v)
    return 'negligible' if v < 0.1 else 'weak' if v < 0.3 else 'moderate' if v < 0.5 else 'strong'

# 1. Numbers
pr, pr_p = stats.pearsonr(c[EXP], c[SAL])
sr, sr_p = stats.spearmanr(c[EXP], c[SAL])
lr, lr_p = stats.spearmanr(c['Level_Code'], c[SAL])
el_lr, el_p = stats.spearmanr(c[EXP], c['Level_Code'])
eta_role = eta(c['Job Role'], c[SAL])
eta_level = eta(c['Job Level'], c[SAL])
anova_role = stats.f_oneway(*[g.values for _, g in c.groupby('Job Role')[SAL]])
anova_level = stats.f_oneway(*[g.values for _, g in c.groupby('Job Level')[SAL]])

summary = pd.DataFrame([
    ['Experience (Years)', 'Pearson r', pr, pr ** 2, pr_p],
    ['Experience (Years)', 'Spearman rho', sr, sr ** 2, sr_p],
    ['Job Level (ordinal)', 'Spearman rho', lr, lr ** 2, lr_p],
    ['Job Level (categorical)', 'Eta (ANOVA)', eta_level, eta_level ** 2, anova_level.pvalue],
    ['Job Role (categorical)', 'Eta (ANOVA)', eta_role, eta_role ** 2, anova_role.pvalue],
], columns=['Factor', 'Method', 'Coefficient', 'Variance explained', 'p-value'])
summary['Strength'] = summary['Coefficient'].apply(strength)
summary['Significant (p<0.05)'] = summary['p-value'] < 0.05
display(summary.round(4))
print(f"Experience vs Job Level overlap (Spearman): {el_lr:.3f} ({strength(el_lr)})")

# 2. Correlation heatmap (numeric / ordinal variables)
corr = c[[EXP, 'Level_Code', SAL]].rename(columns={'Level_Code': 'Job Level (code)'}).corr(method='spearman')
fig1 = px.imshow(corr, text_auto='.2f', color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
                 title='Spearman Correlation: Experience, Job Level, Salary')
fig1.update_layout(height=450, template='plotly_white')
fig1.show()

# 3. Which factor explains salary the most? (variance explained)
bar = pd.DataFrame({
    'Factor': ['Experience (Years)', 'Job Level', 'Job Role'],
    'Variance explained (%)': [pr ** 2 * 100, eta_level ** 2 * 100, eta_role ** 2 * 100],
}).sort_values('Variance explained (%)')
fig2 = px.bar(bar, x='Variance explained (%)', y='Factor', orientation='h',
              text=bar['Variance explained (%)'].round(2).astype(str) + '%',
              title='How much of the Salary variation does each factor explain?',
              color='Variance explained (%)', color_continuous_scale='Blues')
fig2.update_layout(template='plotly_white', height=350, coloraxis_showscale=False)
fig2.show()

# 4. Scatter: Experience vs Salary, colored by Job Level
plot_df = c.sample(min(len(c), 5000), random_state=42)
fig3 = px.scatter(plot_df, x=EXP, y=SAL, color='Job Level', opacity=0.5,
                  category_orders={'Job Level': LEVELS}, trendline='ols',
                  hover_data=['Job Role'],
                  title='Experience vs Salary (trendline per Job Level)')
fig3.update_layout(template='plotly_white', height=550)
fig3.show()

# 5. Salary distribution by Job Level
fig4 = px.box(c, x='Job Level', y=SAL, color='Job Level', category_orders={'Job Level': LEVELS},
              title='Salary Distribution by Job Level')
fig4.update_layout(template='plotly_white', height=500, showlegend=False)
fig4.show()

Level order (0 = lowest): ['Junior', 'Mid-Level', 'Senior', 'Lead', 'Principal / Staff']


,Factor,Method,Coefficient,Variance explained,p-value,Strength,Significant (p<0.05)
0,Experience (Years),Pearson r,0.9011,0.8119,0.0,strong,True
1,Experience (Years),Spearman rho,0.9222,0.8504,0.0,strong,True
2,Job Level (ordinal),Spearman rho,0.9491,0.9009,0.0,strong,True
3,Job Level (categorical),Eta (ANOVA),0.9350,0.8742,0.0,strong,True
4,Job Role (categorical),Eta (ANOVA),0.2023,0.0409,0.0,weak,True


Experience vs Job Level overlap (Spearman): 0.973 (strong)


In [11]:
# ==============================================================================
# JOBS WITHOUT CERT / EXPERIENCE REQUIREMENT + SELF-TAUGHT ACCEPTED
# ==============================================================================
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import ipywidgets as widgets
from IPython.display import display

pio.renderers.default = "colab"

SAL, EXP, CERT, EDU = 'Salary (USD/Year)', 'Experience (Years)', 'Certifications', 'Education'
SELF = 'Bootcamp + Self-Taught'

a = clean.copy()
a[EXP] = pd.to_numeric(a[EXP], errors='coerce')
a = a.dropna(subset=[EXP])

cert_txt = a[CERT].astype(str).str.strip().str.lower()
a['No_Cert'] = a[CERT].isna() | cert_txt.isin(['', 'nan', 'none', 'n/a', 'na', '-'])
a['No_Exp'] = a[EXP] == 0
a['Self_Taught'] = a[EDU].astype(str).str.strip().str.lower() == SELF.lower()

FLAGS = {'No_Cert': 'No cert', 'No_Exp': 'No exp', 'Self_Taught': 'Self-taught OK'}
ALL_REQ = 'All required (cert + exp + formal edu)'

def label(r):
    parts = [txt for col, txt in FLAGS.items() if r[col]]
    return ' + '.join(parts) if parts else ALL_REQ

a['Requirement Group'] = a.apply(label, axis=1)
a['Flags'] = a[list(FLAGS)].sum(axis=1)
ORDER = (a.groupby('Requirement Group')['Flags'].first()
           .sort_values(ascending=False, kind='stable').index.tolist())

# 1. Per-requirement summary: how many jobs do NOT require each item
n = len(a)
rows = [[txt, int(a[col].sum()), a[col].mean() * 100] for col, txt in FLAGS.items()]
rows.append(['At least one of the 3', int((a.Flags >= 1).sum()), (a.Flags >= 1).mean() * 100])
rows.append(['At least two of the 3', int((a.Flags >= 2).sum()), (a.Flags >= 2).mean() * 100])
rows.append(['All 3 (fully open-door)', int((a.Flags == 3).sum()), (a.Flags == 3).mean() * 100])
rows.append(['None (everything required)', int((a.Flags == 0).sum()), (a.Flags == 0).mean() * 100])
summary = pd.DataFrame(rows, columns=['Condition', 'Jobs', 'Share (%)']).round({'Share (%)': 1})
display(summary)

fig1 = px.bar(summary.iloc[::-1], x='Jobs', y='Condition', orientation='h', color='Condition',
              text=summary.iloc[::-1].apply(lambda r: f"{r.Jobs:,} ({r['Share (%)']}%)", axis=1),
              title='Jobs That Do NOT Require Cert / Experience / Formal Education')
fig1.update_traces(textposition='outside')
fig1.update_layout(template='plotly_white', showlegend=False, height=450, yaxis_title='',
                   xaxis_title='Number of Job Postings', margin=dict(l=40, r=90, t=60, b=40))
fig1.update_xaxes(range=[0, summary['Jobs'].max() * 1.25])
fig1.show()

# 2. Every combination (exact groups)
combo = (a['Requirement Group'].value_counts().reindex(ORDER).rename_axis('Requirement Group')
           .reset_index(name='Jobs'))
combo['Share (%)'] = (combo['Jobs'] / n * 100).round(1)
display(combo)
fig2 = px.bar(combo.iloc[::-1], x='Jobs', y='Requirement Group', orientation='h', color='Requirement Group',
              text=combo.iloc[::-1].apply(lambda r: f"{r.Jobs:,} ({r['Share (%)']}%)", axis=1),
              title='Job Postings by Exact Requirement Combination')
fig2.update_traces(textposition='outside')
fig2.update_layout(template='plotly_white', showlegend=False, yaxis_title='', xaxis_title='Number of Job Postings',
                   height=max(400, 80 + 45 * len(combo)), margin=dict(l=40, r=90, t=60, b=40))
fig2.update_xaxes(range=[0, combo['Jobs'].max() * 1.25])
fig2.show()

# 3. By Job Role / Job Level: % of postings that don't require each item
def by_dim(dim, title):
    g = a.groupby(dim).agg(Jobs=('Flags', 'size'), **{txt: (col, 'mean') for col, txt in FLAGS.items()})
    g['All 3'] = a.assign(x=a.Flags == 3).groupby(dim)['x'].mean()
    g = g.sort_values('All 3', ascending=True)
    pct = (g.drop(columns='Jobs') * 100).reset_index().melt(id_vars=dim, var_name='Condition', value_name='Percent')
    pct = pct.merge(g[['Jobs']].reset_index(), on=dim)
    fig = px.bar(pct, x='Percent', y=dim, color='Condition', barmode='group', orientation='h',
                 category_orders={dim: g.index.tolist()}, custom_data=['Jobs'], title=title)
    fig.update_traces(hovertemplate='%{y}<br>%{fullData.name}: %{x:.1f}% (of %{customdata[0]} postings)<extra></extra>')
    fig.update_layout(template='plotly_white', xaxis_title='% of postings', yaxis_title='',
                      height=max(450, 100 + 55 * len(g)), legend=dict(orientation='h', y=1.06, x=0),
                      margin=dict(t=110, b=40, l=40, r=40))
    fig.show()

by_dim('Job Role', 'Open-door Share by Job Role (sorted by "All 3")')
by_dim('Job Level', 'Open-door Share by Job Level (sorted by "All 3")')

# 4. Separate section: filter the actual jobs
ANY = 'Any'
opt = [ANY, 'Yes (not required / accepted)', 'No (required / not accepted)']
w_cert = widgets.Dropdown(options=opt, description='No cert:', layout=widgets.Layout(width='380px'))
w_exp = widgets.Dropdown(options=opt, description='No exp:', layout=widgets.Layout(width='380px'))
w_self = widgets.Dropdown(options=opt, description='Self-taught:', layout=widgets.Layout(width='380px'))
w_role = widgets.Dropdown(options=[ANY] + sorted(a['Job Role'].unique()), description='Job Role:',
                          layout=widgets.Layout(width='380px'))
w_n = widgets.Dropdown(options=[10, 25, 50, 100], value=25, description='Rows:', layout=widgets.Layout(width='200px'))
out = widgets.Output()

def show(_=None):
    with out:
        out.clear_output(wait=True)
        s = a
        for w, col in [(w_cert, 'No_Cert'), (w_exp, 'No_Exp'), (w_self, 'Self_Taught')]:
            if w.value.startswith('Yes'): s = s[s[col]]
            elif w.value.startswith('No'): s = s[~s[col]]
        if w_role.value != ANY:
            s = s[s['Job Role'] == w_role.value]
        print(f"Matching postings: {len(s):,}\n")
        if s.empty: return
        print("Count by Job Role x Job Level:")
        display(s.groupby(['Job Role', 'Job Level']).size().unstack(fill_value=0))
        cols = [c for c in ['Job Role', 'Job Level', EXP, CERT, EDU, SAL, 'Company Name', 'City', 'Country', 'Date Posted']
                if c in s.columns]
        print(f"\nSample postings (top {w_n.value}):")
        display(s[cols].head(w_n.value))

for w in (w_cert, w_exp, w_self, w_role, w_n):
    w.observe(show, names='value')
display(widgets.VBox([widgets.HBox([w_cert, w_exp]), widgets.HBox([w_self, w_role]), w_n]), out)
show()

,Condition,Jobs,Share (%)
0,No cert,5002,25.0
1,No exp,1346,6.7
2,Self-taught OK,3425,17.1
3,At least one of the 3,8460,42.3
4,At least two of the 3,1265,6.3
5,All 3 (fully open-door),48,0.2
6,None (everything required),11540,57.7


,Requirement Group,Jobs,Share (%)
0,No cert + No exp + Self-taught OK,48,0.2
1,No cert + No exp,241,1.2
2,No cert + Self-taught OK,815,4.1
3,No exp + Self-taught OK,161,0.8
4,No cert,3898,19.5
5,No exp,896,4.5
6,Self-taught OK,2401,12.0
7,All required (cert + exp + formal edu),11540,57.7


Output()

In [12]:
# ==============================================================================
# DO CERTIFICATIONS / EDUCATION / SELF-TAUGHT / NO-EXP AFFECT SALARY?
# ==============================================================================
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
from scipy import stats
import statsmodels.formula.api as smf

pio.renderers.default = "colab"

SAL, EXP, CERT, CNT, EDU = ('Salary (USD/Year)', 'Experience (Years)', 'Certifications',
                            'Certifications Count', 'Education')
SELF = 'Bootcamp + Self-Taught'

# ---------- 1. Prepare data ----------
b = clean[['Job Role', 'Job Level', EXP, CERT, CNT, EDU, SAL]].copy()
b[SAL] = pd.to_numeric(b[SAL], errors='coerce')
b[EXP] = pd.to_numeric(b[EXP], errors='coerce')
b[CNT] = pd.to_numeric(b[CNT], errors='coerce')
b = b.dropna(subset=[EXP, SAL, 'Job Role', 'Job Level', EDU])
b = b[b[SAL] > 0].copy()
for c in ['Job Role', 'Job Level', EDU]:
    b[c] = b[c].astype(str).str.strip()

# Level order junior -> senior (edit if your names differ)
ORDER = ['Junior', 'Mid-Level', 'Senior', 'Lead', 'Principal / Staff']
LEVELS = [l for l in ORDER if l in set(b['Job Level'])] + \
         sorted(set(b['Job Level']) - set(ORDER))

cert_txt = b[CERT].astype(str).str.strip().str.lower()
b['No_Cert'] = b[CERT].isna() | cert_txt.isin(['', 'nan', 'none', 'n/a', 'na', '-'])
b['No_Exp'] = b[EXP] == 0
b['Self_Taught'] = b[EDU].str.lower() == SELF.lower()
b['Has_Cert'] = (~b['No_Cert']).astype(int)
b['Has_Exp'] = (~b['No_Exp']).astype(int)
b['SelfTaught'] = b['Self_Taught'].astype(int)
b['log_sal'] = np.log(b[SAL])

print(f"Rows used: {len(b):,}")
print("Check - Certifications Count when Certifications is empty:")
print(b.groupby('No_Cert')[CNT].describe()[['count', 'mean', 'min', 'max']].round(2))

FACTORS = {'No cert': 'No_Cert', 'No experience': 'No_Exp', 'Self-taught / Bootcamp': 'Self_Taught'}

# ---------- 2. RAW comparison (NOT controlled -> can be misleading) ----------
rows = []
for name, col in FACTORS.items():
    x, y = b.loc[b[col], SAL], b.loc[~b[col], SAL]
    _, p = stats.mannwhitneyu(x, y, alternative='two-sided')
    pooled = np.sqrt((x.var() + y.var()) / 2)
    rows.append({'Factor': name, 'n (yes)': len(x), 'n (no)': len(y),
                 'Mean yes ($)': x.mean(), 'Mean no ($)': y.mean(),
                 'Diff (%)': (x.mean() / y.mean() - 1) * 100,
                 "Cohen's d": (x.mean() - y.mean()) / pooled, 'p-value': p})
raw = pd.DataFrame(rows).round(3)
display(raw)

fig = px.bar(raw, x='Factor', y='Diff (%)', text=raw['Diff (%)'].map('{:+.1f}%'.format),
             color='Factor', title='RAW salary difference when the job does NOT require the item (yes vs no)')
fig.update_traces(textposition='outside')
fig.add_hline(y=0, line_color='gray')
fig.update_layout(template='plotly_white', showlegend=False, height=420, yaxis_title='Salary difference (%)')
fig.show()

# ---------- 3. Why raw can mislead: who are these jobs? ----------
mix = pd.DataFrame({
    name: b.groupby(col).apply(lambda g: pd.Series({
        'Avg experience (yrs)': g[EXP].mean(),
        '% Junior': (g['Job Level'] == LEVELS[0]).mean() * 100}), include_groups=False).loc[True]
    for name, col in FACTORS.items()}).T.round(2)
print("\nProfile of jobs that do NOT require each item:")
display(mix)

# ---------- 4. Within the SAME Job Level ----------
for name, col in FACTORS.items():
    g = (b.groupby(['Job Level', col])[SAL].agg(['mean', 'count']).reset_index())
    g['Group'] = np.where(g[col], f'{name}: YES', f'{name}: NO')
    fig = px.bar(g, x='Job Level', y='mean', color='Group', barmode='group', custom_data=['count'],
                 category_orders={'Job Level': LEVELS},
                 title=f'Average salary by Job Level - {name} (same level comparison)')
    fig.update_traces(hovertemplate='%{x}<br>%{fullData.name}<br>Avg: $%{y:,.0f}<br>Jobs: %{customdata[0]}<extra></extra>')
    fig.update_layout(template='plotly_white', height=450, yaxis_title='Average Salary (USD/Year)')
    fig.show()

# ---------- 5. Controlled regression (log salary -> coefficients read as % effect) ----------
r = b.rename(columns={'Job Role': 'Role', 'Job Level': 'Level', EXP: 'Exp', CNT: 'CertCount'})
terms = {'Has_Cert': 'Has certification', 'Has_Exp': 'Requires experience (>0 yrs)',
         'SelfTaught': 'Self-taught / Bootcamp'}
f_base = 'log_sal ~ Has_Cert + Has_Exp + SelfTaught'
models = {
    'M1 No controls': smf.ols(f_base, r).fit(cov_type='HC3'),
    'M2 + Level + Role': smf.ols(f_base + ' + C(Level) + C(Role)', r).fit(cov_type='HC3'),
    'M3 + Level + Role + Experience (yrs)': smf.ols(f_base + ' + C(Level) + C(Role) + Exp', r).fit(cov_type='HC3'),
}

def tidy(model, mname, tmap):
    ci = model.conf_int()
    out = []
    for v, label in tmap.items():
        e, lo, hi = [(np.exp(z) - 1) * 100 for z in (model.params[v], ci.loc[v, 0], ci.loc[v, 1])]
        out.append({'Model': mname, 'Factor': label, 'Effect (%)': e, 'CI low (%)': lo,
                    'CI high (%)': hi, 'p-value': model.pvalues[v],
                    'plus': hi - e, 'minus': e - lo})
    return out

eff = pd.DataFrame([row for m, mod in models.items() for row in tidy(mod, m, terms)])
eff['Significant'] = eff['p-value'] < 0.05
display(eff.drop(columns=['plus', 'minus']).round(3))
print({k: round(m.rsquared, 3) for k, m in models.items()}, "<- R²")

fig = px.scatter(eff, x='Effect (%)', y='Factor', color='Model', error_x='plus', error_x_minus='minus',
                 title='Salary effect (%) with 95% CI - dots touching 0 = no clear effect')
fig.add_vline(x=0, line_dash='dash', line_color='gray')
fig.update_traces(marker=dict(size=10))
fig.update_layout(template='plotly_white', height=450, legend=dict(orientation='h', y=1.18, x=0))
fig.show()

# ---------- 6. Education (every value) + number of certifications ----------
edu_vals = r[EDU].value_counts().index.tolist()          # most common first = reference group
code = {v: f'E{i}' for i, v in enumerate(edu_vals)}
r['EduCode'] = r[EDU].map(code)
m_edu = smf.ols("log_sal ~ C(EduCode, Treatment(reference='E0')) + C(Level) + C(Role) + Exp",
                r).fit(cov_type='HC3')
tmap = {f"C(EduCode, Treatment(reference='E0'))[T.{c}]": f'{v}' for v, c in code.items() if c != 'E0'}
edu_eff = pd.DataFrame(tidy(m_edu, f'vs "{edu_vals[0]}"', tmap))

print(f'\nEducation effect on salary, SAME level/role/experience, reference group = "{edu_vals[0]}":')
display(edu_eff.drop(columns=['plus', 'minus', 'Model']).round(3))
fig = px.scatter(edu_eff, x='Effect (%)', y='Factor', error_x='plus', error_x_minus='minus',
                 title=f'Education effect on salary (reference: {edu_vals[0]}) - same level, role, experience')
fig.add_vline(x=0, line_dash='dash', line_color='gray')
fig.update_traces(marker=dict(size=10))
fig.update_layout(template='plotly_white', height=max(350, 130 + 50 * len(edu_eff)))
fig.show()

m_cnt = smf.ols('log_sal ~ CertCount + C(Level) + C(Role) + Exp', r).fit(cov_type='HC3')
e = (np.exp(m_cnt.params['CertCount']) - 1) * 100
print(f"\nEach extra certification (same level/role/experience): {e:+.2f}% salary, "
      f"p = {m_cnt.pvalues['CertCount']:.4f}")

# ---------- 7. Auto conclusion ----------
print("\n=== CONCLUSION (model M3: same level, role and experience) ===")
for _, row in eff[eff['Model'].str.startswith('M3')].iterrows():
    crosses0 = row['CI low (%)'] <= 0 <= row['CI high (%)']
    if row['p-value'] >= 0.05 or crosses0:
        verdict = 'NO clear effect on salary'
    elif abs(row['Effect (%)']) < 1:
        verdict = f"statistically significant but tiny ({row['Effect (%)']:+.2f}%)"
    else:
        verdict = f"{'higher' if row['Effect (%)'] > 0 else 'lower'} salary by {abs(row['Effect (%)']):.1f}%"
    print(f"- {row['Factor']}: {verdict} (p={row['p-value']:.3f})")

Rows used: 20,000
Check - Certifications Count when Certifications is empty:
           count  mean  min  max
No_Cert                         
False    14998.0  1.91  1.0  4.0
True      5002.0  0.00  0.0  0.0


,Factor,n (yes),n (no),Mean yes ($),Mean no ($),Diff (%),Cohen's d,p-value
0,No cert,5002,14998,138973.511,137390.419,1.152,0.023,0.129
1,No experience,1346,18654,60516.345,143361.853,-57.788,-1.676,0.000
2,Self-taught / Bootcamp,3425,16575,135061.752,138349.351,-2.376,-0.048,0.030



Profile of jobs that do NOT require each item:


,Avg experience (yrs),% Junior
No cert,6.73,19.11
No experience,0.00,100.00
Self-taught / Bootcamp,6.74,19.15


,Model,Factor,Effect (%),CI low (%),CI high (%),p-value,Significant
0,M1 No controls,Has certification,-0.348,-1.841,1.168,0.651,False
1,M1 No controls,Requires experience (>0 yrs),114.953,112.402,117.535,0.000,True
2,M1 No controls,Self-taught / Bootcamp,-2.764,-4.425,-1.074,0.001,True
3,M2 + Level + Role,Has certification,-0.106,-0.529,0.318,0.623,False
4,M2 + Level + Role,Requires experience (>0 yrs),0.117,-0.856,1.100,0.814,False
5,M2 + Level + Role,Self-taught / Bootcamp,0.090,-0.413,0.595,0.727,False
6,M3 + Level + Role + Experience (yrs),Has certification,-0.105,-0.527,0.320,0.629,False
7,M3 + Level + Role + Experience (yrs),Requires experience (>0 yrs),0.471,-0.531,1.484,0.358,False
8,M3 + Level + Role + Experience (yrs),Self-taught / Bootcamp,0.084,-0.418,0.590,0.743,False


{'M1 No controls': np.float64(0.143), 'M2 + Level + Role': np.float64(0.932), 'M3 + Level + Role + Experience (yrs)': np.float64(0.932)} <- R²



Education effect on salary, SAME level/role/experience, reference group = "Bootcamp + Self-Taught":


,Factor,Effect (%),CI low (%),CI high (%),p-value
0,B.Sc. Computer Science,-0.059,-0.734,0.621,0.865
1,B.Sc. Information Technology,-0.070,-0.832,0.697,0.857
2,B.Eng. Software Engineering,0.251,-0.638,1.149,0.581
3,B.Eng. Electronics & Communication,-0.220,-1.358,0.930,0.706
4,M.Sc. Artificial Intelligence,0.851,-0.590,2.313,0.248
5,B.Sc. Software Engineering,0.385,-0.894,1.679,0.557
6,Ph.D. Artificial Intelligence,1.207,-0.184,2.617,0.089
7,M.Sc. Computer Science,0.343,-0.875,1.576,0.582
8,M.Sc. Data Science,1.703,0.265,3.162,0.020
9,Associate Degree in Computer Science,-0.661,-1.947,0.641,0.318



Each extra certification (same level/role/experience): -0.09% salary, p = 0.2453

=== CONCLUSION (model M3: same level, role and experience) ===
- Has certification: NO clear effect on salary (p=0.629)
- Requires experience (>0 yrs): NO clear effect on salary (p=0.358)
- Self-taught / Bootcamp: NO clear effect on salary (p=0.743)
